# 02. Outlier Detection & Treatment
## Identify outliers using IQR method → Flag + Log Transform approach

**Objective**:
- Detect outliers per indicator (IQR method: Q1 - 1.5×IQR, Q3 + 1.5×IQR)
- Flag outliers in a separate column
- Apply log transformation (not deletion)
- Assumption: World Bank data is accurate, no measurement errors

**Output**:
- Outlier flags per indicator
- Log-transformed versions for ML algorithms
- Outlier summary report

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load consolidated data
data_dir = Path('./outputs')
consolidated = pd.read_csv(data_dir / 'consolidated_raw_data.csv', index_col=0)

print(f"✓ Loaded: {consolidated.shape}")

✓ Loaded: (266, 168)


## 1. Detect Outliers using IQR Method

In [4]:
# Extract indicators
indicators_map = {}
for col in consolidated.columns:
    if '_YR' in col:
        parts = col.rsplit('_YR', 1)
        indicator = parts[0]
        if indicator not in indicators_map:
            indicators_map[indicator] = []
        indicators_map[indicator].append(col)

# Outlier detection per indicator
print("\n" + "="*80)
print("OUTLIER DETECTION (IQR Method)")
print("="*80 + "\n")

outlier_flags = pd.DataFrame(index=consolidated.index)
outlier_summary = []

for indicator, cols in sorted(indicators_map.items()):
    # Get data for this indicator
    data_subset = consolidated[cols]
    
    # Calculate IQR
    Q1 = data_subset.quantile(0.25)
    Q3 = data_subset.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Flag outliers
    outlier_mask = ((data_subset < lower_bound) | (data_subset > upper_bound)).any(axis=1)
    outlier_flags[f"{indicator}_outlier"] = outlier_mask
    
    # Count
    outlier_count = outlier_mask.sum()
    outlier_pct = (outlier_count / len(consolidated)) * 100
    
    # Get min/max
    data_clean = data_subset.dropna().values.flatten()
    min_val = data_clean.min() if len(data_clean) > 0 else np.nan
    max_val = data_clean.max() if len(data_clean) > 0 else np.nan
    
    outlier_summary.append({
        'Indicator': indicator,
        'Outliers': outlier_count,
        'Pct_%': round(outlier_pct, 1),
        'Min': round(min_val, 2),
        'Max': round(max_val, 2),
        'IQR_Lower': round(lower_bound.mean(), 2),
        'IQR_Upper': round(upper_bound.mean(), 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))
print(f"\nTotal outlier flags: {outlier_flags.sum().sum()}")


OUTLIER DETECTION (IQR Method)

           Indicator  Outliers  Pct_%     Min       Max  IQR_Lower  IQR_Upper
  Agriculture Sector        11    4.1    0.03     66.03     -18.53      37.65
     Domestic Credit        18    6.8    1.14    301.02     -61.36     160.66
  Electricity Access        53   19.9     NaN       NaN      28.55     142.84
         FDI Inflows        93   35.0 -391.56    452.22      -4.16      10.90
      GDP per capita        40   15.0  250.88 247170.22  -22721.95   42856.18
     Industry Sector        22    8.3    3.05     86.67      -0.26      51.62
Internet Penetration        22    8.3    2.60    100.00     -38.24     125.38
     Services Sector        13    4.9    6.45     91.92      23.63      88.15

Total outlier flags: 272


## 2. Log Transform Numerical Features

In [5]:
# Create log-transformed version for ML algorithms
# Handle negatives by: log(x + constant)

consolidated_log = consolidated.copy()

print("\n" + "="*80)
print("LOG TRANSFORMATION (for ML algorithms)")
print("="*80 + "\n")

for col in consolidated_log.columns:
    if consolidated_log[col].dtype in [np.float64, np.int64]:
        # Remove NaN
        valid_data = consolidated_log[col].dropna()
        
        if len(valid_data) > 0:
            min_val = valid_data.min()
            
            # Handle negative values
            if min_val < 0:
                # Shift by absolute minimum + small constant
                shift = abs(min_val) + 1
                consolidated_log[col] = np.log(consolidated_log[col] + shift)
            elif min_val == 0:
                # Add small constant to avoid log(0)
                consolidated_log[col] = np.log(consolidated_log[col] + 1)
            else:
                # All positive, direct log
                consolidated_log[col] = np.log(consolidated_log[col])

print("✓ Log transformation applied to all numeric columns")
print(f"  Handled negative values with shifting + log(x+1) approach")


LOG TRANSFORMATION (for ML algorithms)

✓ Log transformation applied to all numeric columns
  Handled negative values with shifting + log(x+1) approach


## 3. Save Outlier-Flagged Data

In [6]:
# Save original + logs + flags
consolidated_log.to_csv(data_dir / 'consolidated_log_transformed.csv')
outlier_flags.to_csv(data_dir / 'outlier_flags.csv')
outlier_df.to_csv(data_dir / 'outlier_summary.csv', index=False)

print("\n✓ Saved:")
print(f"  consolidated_log_transformed.csv")
print(f"  outlier_flags.csv")
print(f"  outlier_summary.csv")
print(f"\n→ Next: 03_missing_value_treatment.ipynb (KNN imputation)")


✓ Saved:
  consolidated_log_transformed.csv
  outlier_flags.csv
  outlier_summary.csv

→ Next: 03_missing_value_treatment.ipynb (KNN imputation)
